# Evaluate Custom Metrics using LLMaaJ on DPTA RAG task type

This notebook should be run using Runtime 24.1 with Python 3.11 or later. If you are using Watson Studio and do not see Python 3.11.x in the environment, please update the runtime.

You can evaluate custom metrics for prompt templates and detached prompt templates using the LLM as a Judge (LLMaaJ) evaluator. This allows you to define domain-specific evaluation criteria beyond built-in generative AI quality metrics. With LLMaaJ, a large language model acts as a judge to assess AI-generated responses using customizable grading logic.

The notebook creates a retrieval-augmented generation prompt template asset, configures OpenScale monitoring, and evaluates custom metrics, generative quality metrics, and model health metrics.

Note : User can search for `EDIT THIS` and fill the inputs needed.

### Contents

- [Package installation and dependencies](#package-installation-and-dependencies)
- [Credentials configuration](#Configure-credentials)
- [Demo Dataset](#sample-data-loading-and-preview)
- [Client initialization](#Initialize-Openscale-and-Factsheet-client)
- [Create and configure detached prompt from project](#Create-and-configure-a-detached-prompt-in-the-project)
    - [Defining prompt template](##Defining-the-Prompt-Template)
    - [Create prompt asset](##Create-the-prompt-asset)
    - [Map openscale instance to project](##Map-openscale-instance-to-project)
    - [Create custom monitor definition](##Create-custom-monitor-definition-for-LLMaaJ-evaluator)
    - [Create llm provider](##Create-the-LLM-provider)
    - [Execute prompt setup](##Execute-prompt-setup)
    - [Risk evaluation](##Risk-evaluation)
    - [View metrics](##View-metrics)
- [Create and configure detached prompt from space](#Create-and-configure-detached-prompt-on-space)  
    - [Promote PTA to space](##Promote-PTA-to-space)
    - [Create deployment](##Create-deployment)
    - [Map openscale instance to space](##Map-openscale-instance-to-space)
    - [Execute prompt setup](##Execute-prompt-setup-for-space)
    - [Risk evaluation](###Evaluate-the-prompt-template-subscription-from-space)
    - [View metrics](##View-metrics-for-space-subscription)
    - [See factsheets information](##View-factsheet-information-for-space-subscription)

# Package installation and dependencies

Install the required Python packages and dependencies needed to execute the notebook successfully.

**Note**: you may need to restart the kernel to use updated libraries.

In [1]:
!pip install --upgrade ibm-aigov-facts-client ibm-watson-openscale "setuptools<82.0.0" | tail -n 1

# Configure credentials

Configure the authentication credentials required to access Watson OpenScale and other related services.

- Set `use_cpd = True` when using **Cloud Pak for Data (CPD)**, or `False` when using **IBM Cloud**.

In [2]:
import warnings

warnings.filterwarnings("ignore")

use_cpd = False

if use_cpd:
    CPD_URL = "<EDIT_THIS>"                 # Your CPD URL
    CPD_USERNAME = "<EDIT_THIS>"            # Your CPD username
    CPD_PASSWORD = "<EDIT_THIS>"
    # CPD_API_KEY = "<EDIT_THIS>"           # Either password or api_key is required

else:
    IAM_URL = "https://iam.cloud.ibm.com"       # Your IAM URL, used for generating the IAM token (e.g., https://iam.cloud.ibm.com)
    WML_URL = "<EDIT_THIS>"                     # WML URL (e.g., https://us-south.ml.cloud.ibm.com)
    CLOUD_API_KEY = "<EDIT_THIS>"               # Cloud API key used for authentication
    DATAPLATFORM_URL = "<EDIT_THIS>"            # Data Platform URL (e.g., https://dataplatform.cloud.ibm.com)
    SERVICE_URL = "<EDIT_THIS>"                 # AI OpenScale URL (e.g., https://aiopenscale.cloud.ibm.com)


## Setting the project id

Specify the project where the prompt template asset will be created.

- You can find the **Project ID** in the project settings.
- The ID is in **UUID format** (e.g., `7cf9e5c2-3850-4c29-bbc1-ddca94740c99`).

In [3]:
PROJECT_ID = "<EDIT_THIS>"  # YOUR_PROJECT_ID

## Setting the space id

Specify the deployment space used for production or pre-production monitoring.

- You can find the **Space ID** in the deployment space settings.
- The ID is in **UUID format** (e.g., `4293d6fe-21ed-434b-ae0c-e1ffd676c05e`).
- This value is used in **Section 6** for space-based deployment.

In [4]:
SPACE_ID = "<EDIT_THIS>"  # YOUR_SPACE_ID

### Create the access token

Generates an access token that is used to authenticate requests to the service.

In [5]:
import requests, json


def generate_access_token():
    headers = {}
    iam_access_token = ""
    if use_cpd:
        headers["Content-Type"] = "application/json"
        headers["Accept"] = "application/json"
        data = {"username": CPD_USERNAME, "password": CPD_PASSWORD}
        data = json.dumps(data).encode("utf-8")
        url = CPD_URL + "/icp4d-api/v1/authorize"
        response = requests.post(url=url, data=data, headers=headers, verify=False)
        json_data = response.json()
        iam_access_token = json_data["token"]
    else:
        headers["Content-Type"] = "application/x-www-form-urlencoded"
        headers["Accept"] = "application/json"
        data = {
            "grant_type": "urn:ibm:params:oauth:grant-type:apikey",
            "apikey": CLOUD_API_KEY,
            "response_type": "cloud_iam",
        }
        response = requests.post(
            IAM_URL + "/identity/token", data=data, headers=headers
        )
        json_data = response.json()
        iam_access_token = json_data["access_token"]
    return iam_access_token


iam_access_token = generate_access_token()

# Sample data loading and preview

Loads the evaluation dataset and displays a preview of the data.

In [6]:
!wget "https://raw.githubusercontent.com/IBM/watson-openscale-samples/main/IBM%20Cloud/WML/assets/data/watsonx/rag_state_union_gt.csv"

--2026-04-20 17:41:39--  https://raw.githubusercontent.com/IBM/watson-openscale-samples/main/IBM%20Cloud/WML/assets/data/watsonx/rag_state_union_gt.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24430 (24K) [text/plain]
Saving to: ‘rag_state_union_gt.csv’

rag_state_union_gt. 100%[===================>]  23.86K  --.-KB/s    in 0.03s   

2026-04-20 17:41:39 (840 KB/s) - ‘rag_state_union_gt.csv’ saved [24430/24430]



In [7]:
import pandas as pd

test_data_path = (
    "rag_state_union_gt.csv"
)
llm_data = pd.read_csv(test_data_path)
llm_data = llm_data.head(5)
llm_data

,context1,context2,context3,context4,question,answer,generated_text
0,"Last month, I announced our plan to supercharg...",For that purpose we’ve mobilized American grou...,"If you travel 20 miles east of Columbus, Ohio,...",But cancer from prolonged exposure to burn pit...,What is ARPA-H?,ARPA-H is the Advanced Research Projects Agenc...,ARPA-H stands for Advanced Research Projects A...
1,So let’s not wait any longer. Send it to my de...,"If you travel 20 miles east of Columbus, Ohio,...",When we use taxpayer dollars to rebuild Americ...,It is going to transform America and put us on...,What is the investment of Ford and GM to build...,Ford is investing $11 billion to build electri...,Ford is investing $11 billion to build electri...
2,My plan will cut the cost in half for most fam...,We got more than 130 countries to agree on a g...,And unlike the $2 Trillion tax cut passed in t...,We’re going after the criminals who stole bill...,What is the proposed tax rate for corporations?,The proposed tax rate for corporations is a 15...,15% minimum tax rate for corporations. The gl...
3,"If you travel 20 miles east of Columbus, Ohio,...",So let’s not wait any longer. Send it to my de...,When we use taxpayer dollars to rebuild Americ...,It is going to transform America and put us on...,What is Intel going to build?,Intel is going to build a $20 billion semicond...,semiconductor “mega site” with up to eight sta...
4,So let’s not wait any longer. Send it to my de...,"If you travel 20 miles east of Columbus, Ohio,...",It is going to transform America and put us on...,Vice President Harris and I ran for office wit...,How many electric vehicle charging stations ar...,The document does not provide information on t...,"500,000 Answer: 500,000 electric vehicle charg..."


# Initialize Openscale and Factsheet client

Initialize the client required to manage AI governance and factsheets.

## Create Factsheet client

In [8]:
from sys import version
from ibm_aigov_facts_client import AIGovFactsClient

import inspect

print(inspect.signature(AIGovFactsClient))

if use_cpd:
    from ibm_aigov_facts_client import CloudPakforDataConfig

    creds = CloudPakforDataConfig(
        service_url=CPD_URL, username=CPD_USERNAME, password=CPD_PASSWORD
    )
    facts_client = AIGovFactsClient(
        cloud_pak_for_data_configs=creds,
        container_id=PROJECT_ID,
        container_type="project",
        disable_tracing=True,
    )
else:
    facts_client = AIGovFactsClient(
        api_key=CLOUD_API_KEY,
        region="dallas",  # Setting for other regions (Ex. eu-de, au-syd, etc..)
        container_id=PROJECT_ID,
        container_type="project",
        disable_tracing=True,
    )

(experiment_name: str = None, container_type: Optional[str] = None, container_id: Optional[str] = None, authenticator: Union[ForwardRef('BearerTokenAuthenticator'), ForwardRef('CloudPakForDataAuthenticator'), ForwardRef('IAMAuthenticator'), ForwardRef('MCSPV2Authenticator'), NoneType] = None, api_key: Optional[str] = None, bearer_token: Optional[str] = None, set_as_current_experiment: Optional[bool] = False, enable_autolog: Optional[bool] = True, external_model: Optional[bool] = False, centralized_model: Optional[bool] = False, cloud_pak_for_data_configs: 'CloudPakforDataConfig' = None, disable_tracing: Optional[bool] = False, enable_push_framework: Optional[bool] = False, region: Optional[str] = None, account_id: Optional[str] = None) -> None


## Create the Openscale client

Initialize the watson openscale client used for model monitoring and governance.

In [9]:
from ibm_cloud_sdk_core.authenticators import (
    IAMAuthenticator,
    CloudPakForDataAuthenticator,
)
from ibm_watson_openscale import *
from ibm_watson_openscale.supporting_classes.enums import *
from ibm_watson_openscale.supporting_classes import *

if use_cpd:
    authenticator = CloudPakForDataAuthenticator(
        url=CPD_URL,
        username=CPD_USERNAME,
        password=CPD_PASSWORD,
        disable_ssl_verification=True,
    )
    wos_client = APIClient(
        service_url=CPD_URL,
        authenticator=authenticator,
    )
    data_mart_id = wos_client.service_instance_id
    print(wos_client.version)
else:
    authenticator = IAMAuthenticator(apikey=CLOUD_API_KEY, url=IAM_URL) 
    wos_client = APIClient(
        authenticator=authenticator,
        service_url=SERVICE_URL,
    )
    data_mart_id = wos_client.service_instance_id
    print(wos_client.version)


WOS_SERVICE_INSTANCE_ID = wos_client.service_instance_id

[Warning] No region provided : Using default region as us-south
3.1.5


# Create and configure a detached prompt in the project

Demonstrates how to create and monitor a prompt template asset in a Watson Studio project.
1. Define the prompt template structure.
2. Create the prompt asset in the project.
3. Configure OpenScale monitoring.
4. Set up custom metrics using LLMaaJ (LLM-as-a-Judge).
5. Run the evaluation and view the results.

## Defining the prompt template

Define the prompt template structure that will be used for RAG (Retrieval-Augmented Generation) tasks.

In [10]:
prompt_input = """[INST] <>You are a knowledgeable assistant that answers questions about U.S. policy, government initiatives, and State of the Union addresses. You must provide accurate, factual answers based solely on the provided context documents. If the answer cannot be found in the context, clearly state that you don't have that information.<>

Context Documents:
{context1}
{context2}
{context3}

Instructions:
- Answer questions using ONLY information from the provided context documents
- Provide specific details like numbers, names, and facts when available
- If the answer is not in the context, respond: "I don't have enough information in the provided context to answer that question."
- Keep answers concise and factual
- Do not make assumptions or add information not present in the context
- When multiple contexts contain relevant information, synthesize them coherently

Question: {question} [/INST]

Step 1: Identify relevant information in the provided contexts
Step 2: Extract specific facts, figures, and details that answer the question
Step 3: Formulate a clear, accurate answer based only on the context
"""

## Create the prompt asset

Create a detached prompt template asset in the project.

**Key Parameters:**
| Parameter | Description |
|-----------|-------------|
| **task_id** | Specifies the task type for which the prompt template is created. |
| **model_id** | Identifier of the LLM model used to execute the prompt. |
| **prompt_variables** | Defines the placeholders used inside the prompt template that will be replaced with runtime values. |
| **detached_information** | Contains external model configuration details such as provider information, endpoint URL, and other model connection settings. |

In [11]:
from ibm_aigov_facts_client import DetachedPromptTemplate, PromptTemplate

detached_information = DetachedPromptTemplate(
    prompt_id="detached_prompt",
    model_id="meta-llama/llama-3-70b-instruct",
    model_provider="Facebook",
    model_name="llama-3-70b-instruct",
    model_url="https://us-south.ml.cloud.ibm.com/ml/v1/deployments/insurance_test_deployment/text/generation?version=2021-05-01",
    prompt_url="prompt_url",
    prompt_additional_info={"IBM Cloud Region": "us-east1"},
)

task_id = "retrieval_augmented_generation"
name = "LLMaaJ as custom_monitor Demo"
description = "LLMaaJ as custom_monitor sample notebook"
model_id = "meta-llama/llama-3-70b-instruct"

# Define parameters for PromptTemplate
prompt_variables = {"context1": "", "context2": "", "context3": "", "question": ""}
input = prompt_input
input_prefix = ""
output_prefix = ""

prompt_template = PromptTemplate(
    input=input,
    prompt_variables=prompt_variables,
    input_prefix=input_prefix,
    output_prefix=output_prefix,
)

pta_details = facts_client.assets.create_detached_prompt(
    model_id=model_id,
    task_id=task_id,
    name=name,
    description=description,
    prompt_details=prompt_template,
    detached_information=detached_information,
)
project_pta_id = pta_details.to_dict()["asset_id"]

2026/04/20 17:42:01 INFO : ------------------------------ Detached Prompt Creation Started ------------------------------
2026/04/20 17:42:03 INFO : The detached prompt with ID 2df62e42-8669-4a28-bc5a-b487fe127593 was created successfully in container_id 0b169115-369b-42b0-a459-c11f1e948cef.


## Map openscale instance to project

Link the Watson OpenScale instance to your project.

In [12]:
from ibm_watson_openscale.base_classes import ApiRequestFailure

if use_cpd:
    try:
        wos_client.wos.add_instance_mapping(
            service_instance_id=WOS_SERVICE_INSTANCE_ID, project_id=PROJECT_ID
        )
    except ApiRequestFailure as arf:
        if arf.response.status_code == 409:
            # Instance mapping already exists
            pass
        else:
            raise arf

## Create custom monitor definition for LLMaaJ evaluator
- Defining the configuration required to create a **custom LLMaaJ monitor definition.** 
- LLMaaJ uses a large language model as a **judge** to evaluate the quality of model-generated responses based on predefined prompts and grading scales.

### 1. LLMaaJ metric key configuration fields:

#### **prompt**

- Defines the instruction provided to the LLM judge describing how a response should be evaluated.  
- The prompt contains placeholders that will later be replaced with values from the evaluation dataset during metric computation.

#### **grading_options**

- Defines the scoring scale used by the LLM judge. 
- Each option represents a possible evaluation outcome and is associated with a numeric score that will be recorded as the metric result.

#### **grader_prompt_variables_mapping**

- Maps placeholders used inside the evaluation prompt to the corresponding column names in the dataset.  
- This mapping ensures that the correct dataset values are substituted into the prompt before it is sent to the LLM judge for evaluation.
- Example:

| Prompt Variable | Dataset Column |
|----------------|---------------|
| `{question}` | `question` |
| `{output}` | `answer` |


#### **dataset_type**

- Specifies which dataset is used to compute the metric.
- This field determines the source of records that will be evaluated by the LLM judge when calculating metric scores.
- Supported dataset_types are `"feedback"` or `"payload-logging"`.




These configurations are required for each metric to ensure that the LLM judge has all the required information to accurately evaluate responses for each metric.

In [13]:
# ============================================================
# Answer Completeness Metric
# ============================================================

ANSWER_COMPLETENESS_PROMPT = """
You are an expert grader. Your job is to evaluate the completeness of an AI-generated response based on the user question
**Question:**
{input}
**AI-Generated Response:**
{output}
Compare the above Question to the AI-generated response. You must determine whether the response is complete using the below grading scale.
## Grading Scale:Rate the response as complete, incomplete or partial based on the below criteria:
complete - The response is complete and thoroughly addresses all parts of the question.
partial - The response addresses some parts of the question but is missing key information.
incomplete - The response fails to address the question
"""

ANSWER_COMPLETENESS_GRADING_OPTIONS = [
    {"name": "complete", "value": 1},
    {"name": "partial", "value": 0.5},
    {"name": "incomplete", "value": 0},
]

ANSWER_COMPLETENESS_GRADER_PROMPT_VARIABLE_MAPPING = {
    "input": "question",
    "output": "answer",
}

ANSWER_COMPLETENESS_COMPUTING_DATASET_TYPE = "feedback"


# ============================================================
# Conciseness Metric
# ============================================================

CONCISENESS_PROMPT = """
You are an evaluator assessing the conciseness of a model-generated response.

Conciseness means the response:
- Avoids unnecessary words, repetition, or filler content
- Provides the required information clearly and directly
- Does not include irrelevant explanations

Evaluate the conciseness of the RESPONSE with respect to the QUESTION.

QUESTION:
{question}

RESPONSE:
{response}

Based on the scoring criteria below, select the most appropriate score.

Scoring criteria:
1.0 – Very concise, direct, no unnecessary content
0.75 – Mostly concise with minor verbosity
0.5 – Some unnecessary or repetitive content
0.25 – Very verbose with significant unnecessary information
0.0 – Extremely verbose or mostly irrelevant content
"""

CONCISENESS_GRADING_OPTIONS = [
    {
        "name": "1.0",
        "description": "Very concise and direct with no unnecessary content",
        "value": 1.0,
    },
    {
        "name": "0.75",
        "description": "Mostly concise with minor verbosity",
        "value": 0.75,
    },
    {
        "name": "0.5",
        "description": "Contains some unnecessary or repetitive content",
        "value": 0.5,
    },
    {
        "name": "0.25",
        "description": "Very verbose with significant unnecessary information",
        "value": 0.25,
    },
    {
        "name": "0.0",
        "description": "Extremely verbose or mostly irrelevant content",
        "value": 0.0,
    },
]

CONCISENESS_GRADING_GRADER_PROMPT_VARIABLE_MAPPING = {
    "question": "question",
    "response": "answer",
}

CONCISENESS_COMPUTING_DATASET_TYPE = "feedback"

### 2. Create metrics configuration
- The configuration defines the metrics used by the custom monitor.  
- Each metric is configured with the following components:

| Field | Description |
|------|-------------|
| `name` | Unique identifier for the metric. |
| `description` | Explanation of the metric. |
| `thresholds` | Acceptable score limits used for monitoring alerts. |
| `computation` | Defines the evaluation prompt and grading options used by the LLM judge.|
| `grader_prompt_variables_mapping` | Mapping between dataset columns and prompt variables. |
| `dataset_type` | Dataset used to compute the metric.|


In [14]:
# ============================================================
# Creating Monitor Metrics Configuration Configuration
# ============================================================
MONITOR_METRICS = [
    {
        "name": "answer_completeness",
        "thresholds": {"lower_limit": 0.4},  # Thershold
        "description": "Evaluates the completeness of AI-generated responses",
        "computation": {
            "prompt": ANSWER_COMPLETENESS_PROMPT,
            "grading_options": ANSWER_COMPLETENESS_GRADING_OPTIONS,
        },
        "grader_prompt_variables_mapping": ANSWER_COMPLETENESS_GRADER_PROMPT_VARIABLE_MAPPING,
        "dataset_type": ANSWER_COMPLETENESS_COMPUTING_DATASET_TYPE,
    },
    {
        "name": "conciseness",
        "description": "Evaluates how concise the response is while still providing the necessary information.",
        "computation": {
            "prompt": CONCISENESS_PROMPT,
            "grading_options": CONCISENESS_GRADING_OPTIONS,
        },
        "grader_prompt_variables_mapping": CONCISENESS_GRADING_GRADER_PROMPT_VARIABLE_MAPPING,
        "dataset_type": CONCISENESS_COMPUTING_DATASET_TYPE,
    },
]

### 3. Create custom monitor definition

- The above configurations are used to construct the monitor definition payload.  
- This payload is passed to the OpenScale client to create the custom monitor definition.

| Field | Description |
|------|-------------|
| `CUSTOM_MONITOR_DEFINITION_NAME` | Unique name for the custom monitor definition. |
| `PROBLEM_TYPE` | Specifies the problem type being monitored. |
| `ENABLE_SCHEDULE` | Enables periodic execution of the monitor when set to `True`. |
| `MONITOR_METRICS` | List of LLMaaJ metrics evaluated by the monitor. |
| `DELETE_CUSTOM_MONITOR` | Deletes any existing monitor with the same name before creating a new one. |



In [15]:
CUSTOM_MONITOR_DEFINITION_NAME = "answer_quality_monitor"  # Set custom monitor definition name
PROBLEM_TYPE = "retrieval_augmented_generation"

# ============================================================
# Custom Monitor Definition Configuration
# ============================================================
MONITOR_DEFINITION_CONFIG = {
    "CUSTOM_MONITOR_NAME": CUSTOM_MONITOR_DEFINITION_NAME,
    "PROBLEM_TYPE": PROBLEM_TYPE,
    "ENABLE_SCHEDULE": False,                # Set to True if you want to schedule the monitor
    "MONITOR_METRICS": MONITOR_METRICS,
    "DELETE_CUSTOM_MONITOR": True           # Specify whether to delete existing monitor definition if it already exists
}

# Creating monitor definition
custom_monitor_definition_id = (
    wos_client.custom_monitor.create_llmaj_monitor_definition(
        config=MONITOR_DEFINITION_CONFIG
    )
)
print("custom monitor defition id: ",custom_monitor_definition_id)




 Waiting for end of adding monitor definition answer_quality_monitor 




finished

-------------------------------------------------
 Successfully finished adding monitor definition 
-------------------------------------------------


custom monitor defition id:  answer_quality_monitor


## Create the LLM provider

Configure the LLM that will act as the **"judge"** for evaluating custom metrics.

| Configuration | Description |
|---------------|-------------|
| **Provider type** | Specifies the LLM provider used for evaluation. Supported providers include `watsonx.ai` and `openai`. |
| **Model ID** | The specific model used by the provider to perform the evaluation. |
| **Authentication** | Credentials required to access the provider, such as an API key or username/password. |
| **Project/Space ID** | Required when using `watsonx.ai` models to identify the project or deployment space where the model is hosted. |
| **wml_location** | Specifies the environment where the `watsonx.ai` model is hosted. Supported values include `cpd_local`, `cpd_remote`, `cloud_remote`, and `cloud`. |


In [16]:
LLM_PROVIDER_TYPE = "watsonx.ai"                            # Supported values "watsonx.ai", "openai"
WML_LOCATION = "cloud"                                      # Supported values "cpd_local", "cpd_remote", "cloud_remote", "cloud"

if WML_LOCATION is ["cpd_local", "cpd_remote"]:
    LLM_PROVIDER_CONFIG = {
        "url": CPD_URL,
        "username": CPD_USERNAME,                            # Username is required for cpd
        # "apikey": CPD_API_KEY,
        "password": CPD_PASSWORD,                            # Eitheir apikey or password is required
        "model_id": "granite-3-3-8b-instruct",               # Watsonx.ai model id
        "wml_location": WML_LOCATION,                        # supported values "cpd_local", "cpd_remote"
        "project_id": PROJECT_ID,
        # "space_id": SPACE_ID,                              # space_id or project_id is are required, From the watsonx.ai environment
    }
else:
    LLM_PROVIDER_CONFIG = {
        "apikey": CLOUD_API_KEY,                                   # Watsonx.ai cloud apikey
        "model_id": "meta-llama/llama-3-2-11b-vision-instruct",    # Watsonx.ai cloud model id (e.g., "meta-llama/llama-3-3-70b-instruct")
        "url": WML_URL,                                            # Watsonx.ai cloud url
        "wml_location": WML_LOCATION,                              # Supported values "cloud"
        "project_id": PROJECT_ID,
        # "space_id": SPACE_ID,                                    # space_id or project_id is are required, From the watsonx.ai environment
    }


DELETE_LLM_PROVIDER = True                                  # Specify whether to delete the LLM provider if it already exists

In [17]:
# Creating LLM provider
llm_provider_id = wos_client.custom_monitor.create_llm_provider(
    llm_provider_type=LLM_PROVIDER_TYPE,
    provider_config=LLM_PROVIDER_CONFIG,
    delete_llm_provider=DELETE_LLM_PROVIDER
)
print("llm_provider_id: ", llm_provider_id)

llm_provider_id:  019daace-37f5-7e14-ba63-a242233865a0


## Execute prompt setup

Configure and activate monitoring for the prompt template asset.

In [18]:
label_column = "answer"
context_fields = ["context1", "context2", "context3"]
question_field = "question"
operational_space_id = "development"
problem_type = "retrieval_augmented_generation"
input_data_type = "unstructured_text"


monitors = {
    "generative_ai_quality": {
        "parameters": {"min_sample_size": 5, "metrics_configuration": {}}
    },
    custom_monitor_definition_id: {
        "parameters": {
            "min_records": 5, # Minimum number of records required to run custom metric evaluation.
            "metrics_configuration": {
                "dataset_type": "feedback", # Common dataset_type used for evaluating custom metrics if not specified at the metric level.
                "llm_provider_id": llm_provider_id, # Identifier of the LLM used as the judge for evaluation. 
                "metrics": [
                    {
                        "metric_id": MONITOR_METRICS[0]["name"],
                        "grader_prompt_variables_mapping": MONITOR_METRICS[0][
                            "grader_prompt_variables_mapping"
                        ],
                    },
                    {
                        "metric_id": MONITOR_METRICS[1]["name"],
                        "grader_prompt_variables_mapping": MONITOR_METRICS[1][
                            "grader_prompt_variables_mapping"
                        ],
                    },
                ],
            },
        }
    },
}

response = wos_client.wos.execute_prompt_setup(
    prompt_template_asset_id=project_pta_id,
    project_id=PROJECT_ID,
    label_column=label_column,
    context_fields=context_fields,
    question_field=question_field,
    operational_space_id=operational_space_id,
    problem_type=problem_type,
    input_data_type=input_data_type,
    supporting_monitors=monitors,
    background_mode=False,
)

project_prompt_setup_result = response.result.to_dict()
print("prompt_setup_result: ")
print(json.dumps(project_prompt_setup_result, indent=4))




 Waiting for end of adding prompt setup 2df62e42-8669-4a28-bc5a-b487fe127593 




running.
finished

---------------------------------------------------------------
 Successfully finished setting up prompt template subscription 
---------------------------------------------------------------


prompt_setup_result: 
{
    "prompt_template_asset_id": "2df62e42-8669-4a28-bc5a-b487fe127593",
    "project_id": "0b169115-369b-42b0-a459-c11f1e948cef",
    "service_provider_id": "019d95cb-3a1f-73b6-95f6-a460127d91be",
    "subscription_id": "019daace-456b-70bb-b222-43cc59604e34",
    "mrm_monitor_instance_id": "019daace-6dfe-7e4e-ae65-06bb2da31e96",
    "start_time": "2026-04-20T12:12:16.745479Z",
    "end_time": "2026-04-20T12:12:33.886065Z",
    "status": {
        "state": "FINISHED"
    }
}


#### Reading required IDs from prompt setup response

In [19]:
project_subscription_id = project_prompt_setup_result["subscription_id"]
project_mrm_monitor_instance_id = project_prompt_setup_result["mrm_monitor_instance_id"]
print("subscription_id", project_subscription_id)
print("mrm_monitor_instance_id", project_mrm_monitor_instance_id)

subscription_id 019daace-456b-70bb-b222-43cc59604e34
mrm_monitor_instance_id 019daace-6dfe-7e4e-ae65-06bb2da31e96


#### Show All Monitor Instances for this subscription

Listing the monitors available in the development subscription, along with their current status and configuration details.

In [20]:
wos_client.monitor_instances.show(target_target_id=project_subscription_id)

99c303f8-46bb-4812-9ab3-380d40de9233,active,019daace-456b-70bb-b222-43cc59604e34,subscription,answer_quality_monitor,2026-04-20 12:12:22.694000+00:00,019daace-55ab-756f-947f-5e69bdf2befb
99c303f8-46bb-4812-9ab3-380d40de9233,active,019daace-456b-70bb-b222-43cc59604e34,subscription,mrm,2026-04-20 12:12:28.785000+00:00,019daace-6dfe-7e4e-ae65-06bb2da31e96
99c303f8-46bb-4812-9ab3-380d40de9233,active,019daace-456b-70bb-b222-43cc59604e34,subscription,model_health,2026-04-20 12:12:28.075000+00:00,019daace-6aa1-79ca-8e17-7a0e6adf0fdf
99c303f8-46bb-4812-9ab3-380d40de9233,active,019daace-456b-70bb-b222-43cc59604e34,subscription,generative_ai_quality,2026-04-20 12:12:21.800000+00:00,019daace-520b-723f-a90c-87c54dba2a9e


#### Getting the custom monitor instance ID

Retrieve the unique identifier for the custom monitor instance.
This ID is required to view evaluation metrics and manage the custom monitor.

In [21]:
monitor_instances = wos_client.monitor_instances.list(
    target_target_id=project_subscription_id,
    monitor_definition_id=CUSTOM_MONITOR_DEFINITION_NAME,
).result.to_dict()
project_custom_monitor_id = monitor_instances["monitor_instances"][0]["metadata"]["id"]

print("custom_monitor_id", project_custom_monitor_id)

custom_monitor_id 019daace-55ab-756f-947f-5e69bdf2befb


## Risk evaluation

#### Evaluate the prompt template subscription

Perform a risk assessment on the development subscription using the evaluation dataset.

**Key Parameters**

| Parameter | Description |
|-----------|-------------|
| `test_data_set_name` | Name identifier for the evaluation dataset. |
| `test_data_path` | Path to the CSV file containing the evaluation data. |
| `includes_model_output` | Indicates whether the dataset already contains generated model responses. |
| `background_mode` | Determines whether the evaluation runs asynchronously or waits for completion. |

In [22]:
test_data_set_name = "data"
content_type = "multipart/form-data"
body = {}

# Preparing the test data, removing extra columns
cols_to_remove = ["uid", "doc", "title", "id"]
for col in cols_to_remove:
    if col in llm_data:
        del llm_data[col]
llm_data.to_csv(test_data_path, index=False)

response = wos_client.monitor_instances.mrm.evaluate_risk(
    monitor_instance_id=project_mrm_monitor_instance_id,
    test_data_set_name=test_data_set_name,
    test_data_path=test_data_path,
    content_type=content_type,
    body=body,
    project_id=PROJECT_ID,
    includes_model_output=True,
    background_mode=False,
)




 Waiting for risk evaluation of MRM monitor 019daace-6dfe-7e4e-ae65-06bb2da31e96 




upload_in_progress.
running...
finished

---------------------------------------
 Successfully finished evaluating risk 
---------------------------------------




#### Read the risk evaluation response

Retrieve and display the results of the risk evaluation.

In [23]:
response = wos_client.monitor_instances.mrm.get_risk_evaluation(
    monitor_instance_id=project_mrm_monitor_instance_id, project_id=PROJECT_ID
)
response.result.to_dict()

{'metadata': {'id': '7ed59668-0097-41a4-ab56-e3efba39e8bd',
  'created_at': '2026-04-20T12:12:53.494Z',
  'created_by': 'iam-ServiceId-b317a8da-d926-496e-b0ca-6bcc57f556ae'},
 'entity': {'triggered_by': 'user',
  'parameters': {'evaluation_start_time': '2026-04-20T12:12:39.829624Z',
   'evaluator_user_key': '89b295bd-725f-46c7-aadb-72568666dfa4',
   'facts': {'state': 'finished'},
   'is_auto_evaluated': False,
   'measurement_id': '019daace-cfbb-7228-b60e-58d491233b25',
   'monitors_run_status': [{'monitor_id': 'generative_ai_quality',
     'status': {'state': 'finished'}},
    {'monitor_id': 'model_health', 'status': {'state': 'finished'}},
    {'monitor_id': 'answer_quality_monitor', 'status': {'state': 'finished'}}],
   'project_id': '0b169115-369b-42b0-a459-c11f1e948cef',
   'prompt_template_asset_id': '2df62e42-8669-4a28-bc5a-b487fe127593',
   'user_iam_id': 'IBMid-692000BLQ0',
   'wos_created_deployment_id': '2bc0d7f5-2d3a-5cf6-b056-b2c93e267317',
   'publish_metrics': 'false',


## View metrics

#### Displaying the custom monitor metrics generated through the risk evaluation

View aggregated metrics from the custom monitor evaluation.

In [24]:
wos_client.monitor_instances.show_metrics(
    monitor_instance_id=project_custom_monitor_id, project_id=PROJECT_ID
)

2026-04-20 12:13:09.160788+00:00,answer_completeness,019daacf-0ca8-77ab-ad33-495d81c04be2,0.5,0.4,None,['computed_on:feedback'],answer_quality_monitor,019daace-55ab-756f-947f-5e69bdf2befb,25c781ec-874e-4faf-bb16-8cc2d0bd3f24,subscription,019daace-456b-70bb-b222-43cc59604e34
2026-04-20 12:13:09.160788+00:00,conciseness,019daacf-0ca8-77ab-ad33-495d81c04be2,0.9,None,None,['computed_on:feedback'],answer_quality_monitor,019daace-55ab-756f-947f-5e69bdf2befb,25c781ec-874e-4faf-bb16-8cc2d0bd3f24,subscription,019daace-456b-70bb-b222-43cc59604e34


#### Display record level metrics for custom monitor

View detailed, record-by-record evaluation results.

In [25]:
def get_dataset_id(data_set_type: str, subscription_id):
  data_sets = wos_client.data_sets.list(target_target_id = subscription_id, type = data_set_type).result.data_sets
  data_set_id = None
  if len(data_sets) > 0:
    data_set_id = data_sets[0].metadata.id
  return data_set_id

project_custom_dataset_id = get_dataset_id("custom", project_subscription_id)
print(f"Custom dataset ID: {project_custom_dataset_id}")

Custom dataset ID: 019daace-5890-7fe9-ad8b-b94074bac0ab


In [26]:
wos_client.data_sets.show_records(data_set_id=project_custom_dataset_id)

MRM_3f90d1f9-ae64-4708-ae31-bf81a8931148-0,2026-04-20T12:12:42.526Z,feedback,2026-04-20T12:13:10.524849Z,25c781ec-874e-4faf-bb16-8cc2d0bd3f24,0.5,0.75,019daace-50a8-7e0c-b9f1-53640a2cc77b
MRM_3f90d1f9-ae64-4708-ae31-bf81a8931148-1,2026-04-20T12:12:42.526Z,feedback,2026-04-20T12:13:10.524849Z,25c781ec-874e-4faf-bb16-8cc2d0bd3f24,0.5,0.75,019daace-50a8-7e0c-b9f1-53640a2cc77b
MRM_3f90d1f9-ae64-4708-ae31-bf81a8931148-2,2026-04-20T12:12:42.526Z,feedback,2026-04-20T12:13:10.524849Z,25c781ec-874e-4faf-bb16-8cc2d0bd3f24,0.5,1.0,019daace-50a8-7e0c-b9f1-53640a2cc77b
MRM_3f90d1f9-ae64-4708-ae31-bf81a8931148-3,2026-04-20T12:12:42.526Z,feedback,2026-04-20T12:13:10.524849Z,25c781ec-874e-4faf-bb16-8cc2d0bd3f24,1.0,1.0,019daace-50a8-7e0c-b9f1-53640a2cc77b
MRM_3f90d1f9-ae64-4708-ae31-bf81a8931148-4,2026-04-20T12:12:42.526Z,feedback,2026-04-20T12:13:10.524849Z,25c781ec-874e-4faf-bb16-8cc2d0bd3f24,0.0,1.0,019daace-50a8-7e0c-b9f1-53640a2cc77b


## View Factsheet information for project subscription

Users can navigate to the project to view the facts published in the factsheet.

In [27]:
dataplatform_url = CPD_URL if use_cpd else DATAPLATFORM_URL

factsheets_url = f"{dataplatform_url}/wx/prompt-details/{project_pta_id}/factsheet?context=wx&project_id={PROJECT_ID}"

print(f"User can navigate to the published facts in project {factsheets_url}")

User can navigate to the published facts in project https://dataplatform.cloud.ibm.com/wx/prompt-details/2df62e42-8669-4a28-bc5a-b487fe127593/factsheet?context=wx&project_id=0b169115-369b-42b0-a459-c11f1e948cef


# Create and configure detached prompt on space

Move from development (project) to production (space) environment.

Demonstrates the production deployment workflow:
1. Promote the prompt template asset from project to space
2. Create a deployment for the promoted asset
3. Configure monitoring in the production environment
4. Evaluate the production subscription

## Promote PTA to space

Move the prompt template asset from development project to production space.

In [28]:
headers = {}
headers["Content-Type"] = "application/json"
headers["Accept"] = "*/*"
headers["Authorization"] = "Bearer {}".format(iam_access_token)
verify = True

if use_cpd:
    verify = False

dataplatform_url = CPD_URL if use_cpd else DATAPLATFORM_URL

dataplatform_api_url = dataplatform_url.replace("https://", "https://api.")

url = "{}/v2/assets/{}/promote".format(dataplatform_api_url, project_pta_id)

params = {"project_id": PROJECT_ID}

payload = {"space_id": SPACE_ID}
response = requests.post(
    url, json=payload, headers=headers, params=params, verify=verify
)
json_data = response.json()
space_pta_id = json_data["metadata"]["asset_id"]
print("prompt template asset id from space: ", space_pta_id)

prompt template asset id from space:  efb7690b-ca07-4607-8693-03dd821136e4


## Create deployment

Create a deployment endpoint for the promoted prompt template asset.

In [29]:
from datetime import datetime


DEPLOYMENTS_URL = (CPD_URL if use_cpd else WML_URL) + "/ml/v4/deployments"

payload = {
    "prompt_template": {"id": space_pta_id},
    "detached": {},
    "base_model_id": "meta-llama/llama-3-70b-instruct",
    "description": "rag qa deployment",
    "name": "LLMaaJ as custom_monitor Testing",  # PTA asset name
    "space_id": SPACE_ID,
}

version = datetime.now().strftime(
    "%Y-%m-%d"
)  # The version date for the API of the form YYYY-MM-DD. Example : 2023-07-07
params = {"version": version, "space_id": SPACE_ID}

response = requests.post(
    DEPLOYMENTS_URL, json=payload, headers=headers, params=params, verify=verify
)

json_data = response.json()


if "metadata" in json_data:
    deployment_id = json_data["metadata"]["id"]
    print("Deployment ID: ", deployment_id)
else:
    print(json_data)

Deployment ID:  36e92251-22a8-41c7-9d6e-8084de182fd6


## Map openscale instance to space

Link the OpenScale instance to the deployment space.

In [30]:
from ibm_watson_openscale.base_classes import ApiRequestFailure

if use_cpd:
    try:
        wos_client.wos.add_instance_mapping(
            service_instance_id=WOS_SERVICE_INSTANCE_ID, space_id=SPACE_ID
        )
    except ApiRequestFailure as arf:
        if arf.response.status_code == 409:
            # Instance mapping already exists
            pass
        else:
            raise arf

## Execute prompt Setup for space

Create a production subscription with monitoring for the space deployment.

### Key Configuration

| Parameter | Description |
|-----------|-------------|
| `operational_space_id` | Specifies the deployment environment. Use `pre_production` for staging/testing environments and `production` for live monitoring. |
| `problem_type` | Defines the task type of the Prompt Template Asset (PTA). This must match the PTA task type (e.g., `retrieval_augmented_generation`). |
| `supporting_monitors` | List of monitors enabled for the subscription, including built-in GenAI Quality monitors and custom LLMaaJ monitors. |

In [31]:
label_column = "answer"
context_fields = ["context1", "context2", "context3"]
question_field = "question"
operational_space_id = "production"  # For pre_production flow set this as `pre_production`
problem_type = "retrieval_augmented_generation"
input_data_type = "unstructured_text"

monitors = {
    "generative_ai_quality": {
        "parameters": {"min_sample_size": 5, "metrics_configuration": {}}
    },
    custom_monitor_definition_id: {
        "parameters": {
            "min_records": 5,
            "metrics_configuration": {
                "dataset_type": "feedback",
                "llm_provider_id": llm_provider_id,
                "metrics": [
                    {
                        "metric_id": MONITOR_METRICS[0]["name"],
                        "grader_prompt_variables_mapping": MONITOR_METRICS[0][
                            "grader_prompt_variables_mapping"
                        ],
                    },
                    {
                        "metric_id": MONITOR_METRICS[1]["name"],
                        "grader_prompt_variables_mapping": MONITOR_METRICS[1][
                            "grader_prompt_variables_mapping"
                        ],
                    },
                ],
            },
        }
    },
}


response = wos_client.wos.execute_prompt_setup(
    prompt_template_asset_id=space_pta_id,
    space_id=SPACE_ID,
    deployment_id = deployment_id,
    label_column=label_column,
    context_fields=context_fields,
    question_field=question_field,
    operational_space_id=operational_space_id,
    problem_type=problem_type,
    input_data_type=input_data_type,
    supporting_monitors=monitors,
    background_mode=False,
)

space_prompt_setup_result = response.result.to_dict()
print(json.dumps(space_prompt_setup_result, indent=4))




 Waiting for end of adding prompt setup efb7690b-ca07-4607-8693-03dd821136e4 




running..
finished

---------------------------------------------------------------
 Successfully finished setting up prompt template subscription 
---------------------------------------------------------------


{
    "prompt_template_asset_id": "efb7690b-ca07-4607-8693-03dd821136e4",
    "space_id": "d8b93cb6-2caa-49f4-b08e-6a6b6521adee",
    "service_provider_id": "019daab9-8cd8-7034-bee7-b8c38aa6ff82",
    "subscription_id": "019daacf-9d89-7237-beef-068d963f6610",
    "mrm_monitor_instance_id": "019daacf-cda9-7906-b257-ccef2a2b5eeb",
    "start_time": "2026-04-20T12:13:45.629150Z",
    "end_time": "2026-04-20T12:14:03.907734Z",
    "status": {
        "state": "FINISHED"
    }
}


#### Reading required IDs from prompt setup response

Extract key identifiers from the space subscription creation response.
These IDs are required for risk evaluation and metrics viewing in the space environment.

In [32]:
space_subscription_id = space_prompt_setup_result["subscription_id"]
space_mrm_monitor_instance_id = space_prompt_setup_result["mrm_monitor_instance_id"]
print("subscription_id", space_subscription_id)
print("mrm_monitor_instance_id", space_subscription_id)

subscription_id 019daacf-9d89-7237-beef-068d963f6610
mrm_monitor_instance_id 019daacf-9d89-7237-beef-068d963f6610


#### Show all the monitor instances from this subscription

Verify that all monitors are properly configured and active.

In [33]:
wos_client.monitor_instances.show(target_target_id=space_subscription_id)

99c303f8-46bb-4812-9ab3-380d40de9233,active,019daacf-9d89-7237-beef-068d963f6610,subscription,generative_ai_quality,2026-04-20 12:13:51.954000+00:00,019daacf-b25b-7847-b690-02c679f7efda
99c303f8-46bb-4812-9ab3-380d40de9233,active,019daacf-9d89-7237-beef-068d963f6610,subscription,answer_quality_monitor,2026-04-20 12:13:52.803000+00:00,019daacf-b5fe-787c-a5e6-183b316316f4
99c303f8-46bb-4812-9ab3-380d40de9233,active,019daacf-9d89-7237-beef-068d963f6610,subscription,model_health,2026-04-20 12:13:58.236000+00:00,019daacf-cacc-7bdd-b4da-235baf0aa755
99c303f8-46bb-4812-9ab3-380d40de9233,active,019daacf-9d89-7237-beef-068d963f6610,subscription,mrm,2026-04-20 12:13:58.856000+00:00,019daacf-cda9-7906-b257-ccef2a2b5eeb


#### Getting custom_monitor instance id

Retrieve the custom monitor instance ID for the space subscription.

In [34]:
monitor_instances = wos_client.monitor_instances.list(
    target_target_id=space_subscription_id
).result.to_dict()
space_custom_monitor_id = ""
for monitor_instance in monitor_instances["monitor_instances"]:
    monitor_name = monitor_instance["entity"]["monitor_definition_id"]
    if monitor_name == CUSTOM_MONITOR_DEFINITION_NAME:
        space_custom_monitor_id = monitor_instance["metadata"]["id"]

print("custom_monitor_id", space_custom_monitor_id)

custom_monitor_id 019daacf-b5fe-787c-a5e6-183b316316f4


## Upload records for production space evaluation (Optional for pre_production)

This section is required **only if the user selects a production space**.

### When to Use

| Condition | Description |
|-----------|-------------|
| `operational_space_id = "production"` | This section must be executed to evaluate production monitoring. |
| `operational_space_id = "pre_production"` | This section can be skipped. |

### Add payload data

Retrieve the payload dataset ID for storing scoring requests/responses.

In [35]:
import time
from ibm_watson_openscale.supporting_classes.enums import *

time.sleep(5)
payload_data_set_id = get_dataset_id(DataSetTypes.PAYLOAD_LOGGING, space_subscription_id)

if payload_data_set_id is None:
    print("Payload data set not found. Please check subscription status.")
else:
    print("Payload data set id: ", payload_data_set_id)

Payload data set id:  019daacf-a454-7952-860c-bc257e658f69


Construct payload records from the test data.

In [36]:
import csv

feature_fields = context_fields + [question_field]
prediction = "generated_text"

pl_data = []
prediction_list = []

with open(test_data_path, "r") as csv_file:
    csv_reader = csv.DictReader(csv_file)
    for row in csv_reader:
        request = {"parameters": {"template_variables": {}}}
        for each in feature_fields:
            request["parameters"]["template_variables"][each] = str(row[each])

        predicted_val = row[prediction]
        prediction_list.append(predicted_val)
        response = {"results": [{prediction: predicted_val}]}
        record = {"request": request, "response": response}
        pl_data.append(record)

Store the constructed payload records in OpenScale.

In [37]:
import uuid
from ibm_watson_openscale.supporting_classes.payload_record import PayloadRecord


print("Performing explicit payload logging.")
wos_client.data_sets.store_records(
    data_set_id=payload_data_set_id, request_body=pl_data, background_mode=False
)
time.sleep(5)
pl_records_count = wos_client.data_sets.get_records_count(payload_data_set_id)
print("Number of records in the payload logging table: {}".format(pl_records_count))

Performing explicit payload logging.



 Waiting for end of storing records with request id: 63bf81ad-5f47-4741-a691-33b0a14b497c 




active

---------------------------------------
 Successfully finished storing records 
---------------------------------------


Number of records in the payload logging table: 5


### Add feedback data

Retrieve the feedback dataset ID for storing feedback data.

In [38]:
import time
from ibm_watson_openscale.supporting_classes.enums import *

time.sleep(5)
feedback_data_set_id = get_dataset_id(DataSetTypes.FEEDBACK, space_subscription_id)

if feedback_data_set_id is None:
    print("Feedback data set not found. Please check subscription status.")
else:
    print("Feedback data set id: ", feedback_data_set_id)

Feedback data set id:  019daacf-b133-7dd7-adf3-e5e10ebb91dc


Construct feedback records from the test data.

In [39]:
test_data_content = []
feature_fields = context_fields + [question_field]  # For Alternative Dataset
prediction_list = llm_data["generated_text"].tolist()

for _, row in llm_data.iterrows():
    # Read each row from the DataFrame and add label and prediction values
    result_row = [row[key] for key in feature_fields if key in row]
    result_row.append(row[label_column])
    result_row.append(row["generated_text"])
    test_data_content.append(result_row)

if len(test_data_content) == 5:  # 5 records are there in the downloaded CSV
    print("generated feedback data from DataFrame")
else:
    print(
        "Failed to generate feedback data from DataFrame, kindly verify the DataFrame content"
    )

fields = feature_fields.copy()
fields.append(label_column)
fields.append("_original_prediction")
feedback_data = [{"fields": fields, "values": test_data_content}]

generated feedback data from DataFrame


Store the feedback records in OpenScale.

In [40]:
wos_client.data_sets.store_records(
    data_set_id=feedback_data_set_id, request_body=feedback_data, background_mode=False
)
time.sleep(5)
fb_records_count = wos_client.data_sets.get_records_count(feedback_data_set_id)
time.sleep(10)
print("Number of records in the feedback logging table: {}".format(fb_records_count))




 Waiting for end of storing records with request id: da3a4638-5134-4377-ad07-d83736c21986 




active

---------------------------------------
 Successfully finished storing records 
---------------------------------------


Number of records in the feedback logging table: 5


## Risk evaluation
###  Evaluate the prompt template subscription from space

Perform risk evaluation on the space subscription.

**Note:** Uncomment the pre_production code block if using operational_space_id="pre_production".

In [41]:
####################################################################################
######## For production flow
#####################################################################################
response  = wos_client.monitor_instances.mrm.evaluate_risk(monitor_instance_id=space_mrm_monitor_instance_id,
                                                    body = body,
                                                    space_id = SPACE_ID,
                                                    background_mode = False)




 Waiting for risk evaluation of MRM monitor 019daacf-cda9-7906-b257-ccef2a2b5eeb 




running..
finished

---------------------------------------
 Successfully finished evaluating risk 
---------------------------------------




In [42]:
# #####################################################################################
# ######### For pre_production flow
# ######################################################################################
# response = wos_client.monitor_instances.mrm.evaluate_risk(
#     monitor_instance_id=space_mrm_monitor_instance_id,
#     body=body,
#     test_data_set_name=test_data_set_name,
#     test_data_path=test_data_path,
#     content_type=content_type,
#     includes_model_output=True,
#     space_id=SPACE_ID,
#     background_mode=False,
# )

### Read the risk evaluation response

Retrieve and display space subscription evaluation results.

In [43]:
response = wos_client.monitor_instances.mrm.get_risk_evaluation(
    monitor_instance_id=space_mrm_monitor_instance_id, space_id=SPACE_ID
)
response.result.to_dict()

{'metadata': {'id': 'b8eb492b-3a2a-4821-a49b-9988afb2c063',
  'created_at': '2026-04-20T12:14:59.103Z',
  'created_by': 'iam-ServiceId-b317a8da-d926-496e-b0ca-6bcc57f556ae'},
 'entity': {'triggered_by': 'user',
  'parameters': {'deployment_id': '36e92251-22a8-41c7-9d6e-8084de182fd6',
   'evaluation_start_time': '2026-04-20T12:14:57.653148Z',
   'facts': {'state': 'finished'},
   'measurement_id': '019daad0-ba6e-721b-acfa-87e62d4504f5',
   'monitors_run_status': [{'monitor_id': 'generative_ai_quality',
     'status': {'state': 'finished'}},
    {'monitor_id': 'model_health', 'status': {'state': 'finished'}},
    {'monitor_id': 'answer_quality_monitor', 'status': {'state': 'finished'}}],
   'prompt_template_asset_id': 'efb7690b-ca07-4607-8693-03dd821136e4',
   'prompt_template_details': {'pta_resource_key': '09dbcd75552bcdfa39f1c9207df966e1d73b5e6c8acdc396b315bb981d8a0632'},
   'space_id': 'd8b93cb6-2caa-49f4-b08e-6a6b6521adee',
   'user_iam_id': 'IBMid-692000BLQ0',
   'publish_metrics':

## View metrics for space subscription

#### Displaying the custom monitor metrics generated through the risk evaluation

View aggregated custom metrics for the space subscription.

**Note:** These are the same metrics as the project, but computed for the space environment.

In [44]:
wos_client.monitor_instances.show_metrics(monitor_instance_id=space_custom_monitor_id)

2026-04-20 12:15:11.908792+00:00,answer_completeness,019daad0-ec24-79e7-9a17-ef45a97db5d5,0.5,0.4,None,['computed_on:feedback'],answer_quality_monitor,019daacf-b5fe-787c-a5e6-183b316316f4,bc7af15d-3034-446f-8215-e412bfc78fd8,subscription,019daacf-9d89-7237-beef-068d963f6610
2026-04-20 12:15:11.908792+00:00,conciseness,019daad0-ec24-79e7-9a17-ef45a97db5d5,0.85,None,None,['computed_on:feedback'],answer_quality_monitor,019daacf-b5fe-787c-a5e6-183b316316f4,bc7af15d-3034-446f-8215-e412bfc78fd8,subscription,019daacf-9d89-7237-beef-068d963f6610


#### Display record level metrics for custom monitor

View detailed record-level metrics for the space subscription.

**What you'll see:** Detailed evaluation results for each test record in the space environment.

In [45]:
space_custom_dataset_id = get_dataset_id(DataSetTypes.CUSTOM, space_subscription_id)
print(f"Custom dataset ID: {space_custom_dataset_id}")

Custom dataset ID: 019daacf-b8a5-7eeb-aaea-920f06210846


In [46]:
wos_client.data_sets.show_records(data_set_id=space_custom_dataset_id)

706b46e5-e2c5-471c-ac37-be3f9856a448,2026-04-20T12:14:34.756Z,feedback,2026-04-20T12:15:15.261889Z,bc7af15d-3034-446f-8215-e412bfc78fd8,0.5,0.75,019daacf-b133-7dd7-adf3-e5e10ebb91dc
9e109eea-bfdb-4058-83a3-04810929a358,2026-04-20T12:14:34.756Z,feedback,2026-04-20T12:15:15.261889Z,bc7af15d-3034-446f-8215-e412bfc78fd8,1.0,1.0,019daacf-b133-7dd7-adf3-e5e10ebb91dc
a4055a17-0c9e-4ad8-a724-188255a88f12,2026-04-20T12:14:34.756Z,feedback,2026-04-20T12:15:15.261889Z,bc7af15d-3034-446f-8215-e412bfc78fd8,0.0,0.75,019daacf-b133-7dd7-adf3-e5e10ebb91dc
b5e5d1c5-54ff-4ebd-92ed-68ba4a27883b,2026-04-20T12:14:34.756Z,feedback,2026-04-20T12:15:15.261889Z,bc7af15d-3034-446f-8215-e412bfc78fd8,0.5,0.75,019daacf-b133-7dd7-adf3-e5e10ebb91dc
f4121200-f747-4e1d-a81c-fef47b3b9461,2026-04-20T12:14:34.756Z,feedback,2026-04-20T12:15:15.261889Z,bc7af15d-3034-446f-8215-e412bfc78fd8,0.5,1.0,019daacf-b133-7dd7-adf3-e5e10ebb91dc


## View Factsheet information for space subscription

Users can navigate to the space to view the facts published in the factsheet.

In [47]:
dataplatform_url = CPD_URL if use_cpd else DATAPLATFORM_URL

factsheets_url = f"{dataplatform_url}/ml-runtime/deployments/{deployment_id}/details?space_id={SPACE_ID}"

print(f"User can navigate to the published facts in project {factsheets_url}")

User can navigate to the published facts in project https://dataplatform.cloud.ibm.com/ml-runtime/deployments/36e92251-22a8-41c7-9d6e-8084de182fd6/details?space_id=d8b93cb6-2caa-49f4-b08e-6a6b6521adee


## Congratulations!

You have finished the hands-on lab for IBM Watson OpenScale. You can now navigate to the prompt template asset in your project / space and click on the Evaluate tab to visualise the results on the UI.